# Evaluating the algorithms

Runs each algorithm on its own on the benchmark instances in `test_cases/benchmarks`, and compares it with the optimal cost given with each instance (`*_opt.txt`):

| name | what runs |
|---|---|
| `packing` | `final_solution.run_packing` only |
| `clarke_wright` | `clark_wright_savings.run_clarke_wright` only |
| `rfcs` | `route_first_cluster_second.run_rfcs` only, starting with no solution |
| `full` | `final_solution.find_solution`: packing → Clarke-Wright → RFCS, as `run_algo` runs it |

Each run is a separate process stopped after `TIME_LIMIT` seconds (see `algorithm_runner.py`). Packing and Clarke-Wright finish well before that; RFCS and full always run until they are stopped.

For the step-by-step log at the spec's 30 / 60 / 180 s limits, use `python run_benchmarks.py` instead.

In [1]:
import sys
sys.path.append('../')

import time
from pathlib import Path

import pandas as pd

from algorithm_runner import ALGORITHMS, solve
from run_benchmarks import find_instances, instance_infeasibility, read_instance, read_solution
from test_case_logger import project_score
from utilities import calculate_cost, is_feasible

## Settings

In [23]:
SETS = ['A', 'B', 'P', 'X']                                  # benchmark sets to run # X excluded
SAMPLES_LIMIT = 3                                           # at most this many instances per set (smallest first); None = all
TIME_LIMIT = 30                                             # seconds per run
SELECTED_ALGORITHMS = ['packing', 'clarke_wright', 'rfcs', 'full']     # any of ALGORITHMS

assert set(SELECTED_ALGORITHMS) <= set(ALGORITHMS), f'choose from {ALGORITHMS}'

# rough upper bound: RFCS and full always use the whole time limit
num_instances = sum(min(len(find_instances([s], None)), SAMPLES_LIMIT or float('inf')) for s in SETS)
num_timed = sum(algorithm in ('rfcs', 'full') for algorithm in SELECTED_ALGORITHMS)
print(f'{num_instances} instances, about {num_instances * num_timed * TIME_LIMIT / 60:.0f} min at most')

12 instances, about 12 min at most


## Sanity check: sample from the instructions

In [24]:
from test_cases.sample_from_instructions import cust_demands, dist_matrix, vehicle_capacity, num_vehicles

for algorithm in SELECTED_ALGORITHMS:
    cost, routes, time_taken = solve(algorithm, cust_demands, dist_matrix, vehicle_capacity, num_vehicles, time_limit=2)
    print(f'{algorithm:<14} cost={cost}  routes={routes}  time={time_taken:.2f}s')

packing        cost=26.200000000000003  routes=[[0, 1, 3, 0], [0, 2, 0]]  time=0.02s
clarke_wright  cost=21.8  routes=[[0, 1, 0], [0, 2, 3, 0]]  time=0.02s
rfcs           cost=21.799999999999997  routes=[[0, 3, 2, 0], [0, 1, 0]]  time=2.00s
full           cost=21.799999999999997  routes=[[0, 1, 0], [0, 3, 2, 0]]  time=2.00s


## Run the benchmarks

In [25]:
def run_benchmarks_df(algorithms=SELECTED_ALGORITHMS, sets=SETS, samples_limit=SAMPLES_LIMIT, time_limit=TIME_LIMIT):
    '''Runs every algorithm on the benchmark instances and returns one row per (algorithm, instance).'''
    rows = []
    for s in sets:

        # limit the number of samples if needed
        instances = find_instances([s], None)
        if samples_limit:
            instances = instances[:samples_limit]

        for path in instances:
            # read the test case and provided solution
            cust_demands, dist_matrix, vehicle_cap, num_vehicles = read_instance(path)
            opt_path = path.with_name(path.stem + '_opt.txt')
            optimal_cost = read_solution(opt_path)[0] if opt_path.exists() else None

            instance_info = {
                'set': s, 'instance': path.stem,
                'num_customers': len(cust_demands) - 1, 'vehicle_cap': vehicle_cap, 'num_vehicles': num_vehicles,
                'total_demand': sum(cust_demands),
                'tightness': sum(cust_demands) / (num_vehicles * vehicle_cap),
                'optimal_cost': optimal_cost,
            }

            # instances that can never be solved: no algorithm is run
            impossible = instance_infeasibility(cust_demands, vehicle_cap, num_vehicles)
            if impossible:
                print(f'{path.stem:<16} skipped, instance infeasible: {impossible}')
                continue

            for algorithm in algorithms:
                cost, routes, time_taken = solve(algorithm, cust_demands, dist_matrix, vehicle_cap, num_vehicles, time_limit)

                # feasible only if the routes pass every rule and the saved cost matches them
                feasible = (routes is not None and is_feasible(routes, cust_demands, vehicle_cap, num_vehicles)
                            and abs(calculate_cost(dist_matrix, routes) - cost) < 1e-6)

                rows.append({
                    'algorithm': algorithm, **instance_info,
                    # results
                    'time_taken': time_taken,
                    'cost': cost if feasible else None,
                    'num_routes': len(routes) if routes else None,
                    'gap_pct': 100 * (cost - optimal_cost) / optimal_cost if feasible and optimal_cost else None,
                    'score': project_score(time_taken, time_limit, cost, optimal_cost) if feasible and optimal_cost else 0,
                    'feasible': feasible,
                })
                gap = rows[-1]['gap_pct']
                print(f'{algorithm:<14} {path.stem:<16} gap={gap:7.2f}%  time={time_taken:6.1f}s' if gap is not None
                      else f'{algorithm:<14} {path.stem:<16} infeasible   time={time_taken:6.1f}s', flush=True)
    return pd.DataFrame(rows)


results = run_benchmarks_df()

packing        A-n32-k5         gap=  72.32%  time=   0.0s
clarke_wright  A-n32-k5         gap=   7.40%  time=   0.0s
rfcs           A-n32-k5         gap=   0.00%  time=  30.0s
full           A-n32-k5         gap=   0.00%  time=  30.0s
packing        A-n33-k5         gap=  91.98%  time=   0.0s
clarke_wright  A-n33-k5         gap=   8.32%  time=   0.0s
rfcs           A-n33-k5         gap=   0.00%  time=  30.0s
full           A-n33-k5         gap=   0.00%  time=  30.0s
packing        A-n33-k6         gap=  61.99%  time=   0.0s
clarke_wright  A-n33-k6         gap=   6.87%  time=   0.0s
rfcs           A-n33-k6         gap=   0.00%  time=  30.0s
full           A-n33-k6         gap=   0.00%  time=  30.0s
packing        B-n31-k5         gap=  39.14%  time=   0.0s
clarke_wright  B-n31-k5         gap=   0.74%  time=   0.0s
rfcs           B-n31-k5         gap=   0.00%  time=  30.0s
full           B-n31-k5         gap=   0.00%  time=  30.0s
packing        B-n34-k5         gap=  37.06%  time=   0.

In [26]:
results.head()

,algorithm,set,instance,num_customers,vehicle_cap,num_vehicles,total_demand,tightness,optimal_cost,time_taken,cost,num_routes,gap_pct,score,feasible
0,packing,A,A-n32-k5,31,100,5,410,0.820,784,0.023062,1351.0,5.0,72.321429,0.734426,True
1,clarke_wright,A,A-n32-k5,31,100,5,410,0.820,784,0.021994,842.0,5.0,7.397959,0.964330,True
2,rfcs,A,A-n32-k5,31,100,5,410,0.820,784,30.002281,784.0,5.0,0.000000,0.999974,True
3,full,A,A-n32-k5,31,100,5,410,0.820,784,30.002391,784.0,5.0,0.000000,0.999972,True
4,packing,A,A-n33-k5,32,100,5,446,0.892,661,0.024047,1269.0,5.0,91.981846,0.684974,True


## Comparison

In [32]:
results

,algorithm,set,instance,num_customers,vehicle_cap,num_vehicles,total_demand,tightness,optimal_cost,time_taken,cost,num_routes,gap_pct,score,feasible
0,packing,A,A-n32-k5,31,100,5,410,0.820000,784,0.023062,1351.0,5.0,72.321429,0.734426,True
1,clarke_wright,A,A-n32-k5,31,100,5,410,0.820000,784,0.021994,842.0,5.0,7.397959,0.964330,True
2,rfcs,A,A-n32-k5,31,100,5,410,0.820000,784,30.002281,784.0,5.0,0.000000,0.999974,True
3,full,A,A-n32-k5,31,100,5,410,0.820000,784,30.002391,784.0,5.0,0.000000,0.999972,True
4,packing,A,A-n33-k5,32,100,5,446,0.892000,661,0.024047,1269.0,5.0,91.981846,0.684974,True
5,clarke_wright,A,A-n33-k5,32,100,5,446,0.892000,661,0.023128,716.0,5.0,8.320726,0.960058,True
6,rfcs,A,A-n33-k5,32,100,5,446,0.892000,661,30.002297,661.0,5.0,0.000000,0.999973,True
7,full,A,A-n33-k5,32,100,5,446,0.892000,661,30.002509,661.0,5.0,0.000000,0.999971,True
8,packing,A,A-n33-k6,32,100,6,541,0.901667,742,0.023774,1202.0,6.0,61.994609,0.763374,True
9,clarke_wright,A,A-n33-k6,32,100,6,541,0.901667,742,0.025906,793.0,6.0,6.873315,0.966775,True


In [33]:
# mean cost, time, gap and score per set and algorithm (gap and cost only over feasible runs)
results.groupby(['set', 'algorithm', 'instance', 'optimal_cost'])[['cost', 'time_taken', 'gap_pct', 'score']].mean()

cost  time_taken     gap_pct  \
set algorithm     instance   optimal_cost                                    
A   clarke_wright A-n32-k5   784             842.0    0.021994    7.397959   
                  A-n33-k5   661             716.0    0.023128    8.320726   
                  A-n33-k6   742             793.0    0.025906    6.873315   
    full          A-n32-k5   784             784.0   30.002391    0.000000   
                  A-n33-k5   661             661.0   30.002509    0.000000   
                  A-n33-k6   742             742.0   30.002324    0.000000   
    packing       A-n32-k5   784            1351.0    0.023062   72.321429   
                  A-n33-k5   661            1269.0    0.024047   91.981846   
                  A-n33-k6   742            1202.0    0.023774   61.994609   
    rfcs          A-n32-k5   784             784.0   30.002281    0.000000   
                  A-n33-k5   661             661.0   30.002297    0.000000   
                  A-n33-k6   742             742.0   30.002303    0.000000   
B   clarke_wright B-n31-k5   672             677.0    0.022323    0.744048   
                  B-n34-k5   788             794.0    0.023434    0.761421   
                  B-n35-k5   955             978.0    0.022111    2.408377   
    full          B-n31-k5   672             672.0   30.002330    0.000000   
                  B-n34-k5   788             788.0   30.002413    0.000000   
                  B-n35-k5   955             955.0   30.002335    0.000000   
    packing       B-n31-k5   672             935.0    0.022096   39.136905   
                  B-n34-k5   788            1080.0    0.022842   37.055838   
                  B-n35-k5   955            1501.0    0.022902   57.172775   
    rfcs          B-n31-k5   672             672.0   30.002324    0.000000   
                  B-n34-k5   788             788.0   30.002505    0.000000   
                  B-n35-k5   955             955.0   30.002295    0.000000   
P   clarke_wright P-n16-k8   450               NaN    0.022539         NaN   
                  P-n19-k2   212             237.0    0.021484   11.792453   
                  P-n20-k2   216             234.0    0.021399    8.333333   
    full          P-n16-k8   450             461.0   30.002352    2.444444   
                  P-n19-k2   212             212.0   30.002371    0.000000   
                  P-n20-k2   216             216.0   30.002384    0.000000   
    packing       P-n16-k8   450             540.0    0.022088   20.000000   
                  P-n19-k2   212             273.0    0.023810   28.773585   
                  P-n20-k2   216             279.0    0.022413   29.166667   
    rfcs          P-n16-k8   450               NaN   30.002254         NaN   
                  P-n19-k2   212             212.0   30.002398    0.000000   
                  P-n20-k2   216             216.0   30.003151    0.000000   
X   clarke_wright X-n101-k25 27591             NaN    0.054625         NaN   
                  X-n106-k14 26362         27277.0    0.023624    3.470905   
                  X-n110-k13 14971         16352.0    0.028185    9.224501   
    full          X-n101-k25 27591             NaN   30.002428         NaN   
                  X-n106-k14 26362         26473.0   30.002542    0.421061   
                  X-n110-k13 14971         14971.0   30.002903    0.000000   
    packing       X-n101-k25 27591             NaN    0.052778         NaN   
                  X-n106-k14 26362         38617.0    0.023233   46.487368   
                  X-n110-k13 14971         35762.0    0.023907  138.875159   
    rfcs          X-n101-k25 27591             NaN   30.002604         NaN   
                  X-n106-k14 26362         26515.0   30.002279    0.580381   
                  X-n110-k13 14971         14971.0   30.002483    0.000000   

                                              score  
set algorithm     instance   optimal_cost            
A   clarke_wright A-n32-k5   784     

In [34]:
# feasible runs per set and algorithm
results.groupby(['set', 'algorithm', 'feasible'])['instance'].count()

set  algorithm      feasible
A    clarke_wright  True        3
     full           True        3
     packing        True        3
     rfcs           True        3
B    clarke_wright  True        3
     full           True        3
     packing        True        3
     rfcs           True        3
P    clarke_wright  False       1
                    True        2
     full           True        3
     packing        True        3
     rfcs           False       1
                    True        2
X    clarke_wright  False       1
                    True        2
     full           False       1
                    True        2
     packing        False       1
                    True        2
     rfcs           False       1
                    True        2
Name: instance, dtype: int64

In [35]:
# gap from optimal (%) for each instance, one column per algorithm
results.pivot_table(index=['set', 'instance', 'num_customers'], columns='algorithm', values='gap_pct').round(2)

algorithm                     clarke_wright  full  packing  rfcs
set instance   num_customers                                    
A   A-n32-k5   31                      7.40  0.00    72.32  0.00
    A-n33-k5   32                      8.32  0.00    91.98  0.00
    A-n33-k6   32                      6.87  0.00    61.99  0.00
B   B-n31-k5   30                      0.74  0.00    39.14  0.00
    B-n34-k5   33                      0.76  0.00    37.06  0.00
    B-n35-k5   34                      2.41  0.00    57.17  0.00
P   P-n16-k8   15                       NaN  2.44    20.00   NaN
    P-n19-k2   18                     11.79  0.00    28.77  0.00
    P-n20-k2   19                      8.33  0.00    29.17  0.00
X   X-n106-k14 105                     3.47  0.42    46.49  0.58
    X-n110-k13 109                     9.22  0.00   138.88  0.00

In [36]:
# how often each algorithm gives the lowest cost on an instance (ties count for every algorithm tied)
lowest = results.groupby('instance')['cost'].transform('min')
results[results['cost'] == lowest].groupby('algorithm')['instance'].count()

algorithm
full    11
rfcs     9
Name: instance, dtype: int64

## Save the results

In [ ]:
# log_dir = Path('../logs')
# log_dir.mkdir(exist_ok=True)
# results_path = log_dir / f'notebook_results_{time.strftime("%Y-%m-%d_%H-%M-%S")}.csv'
# results.to_csv(results_path, index=False)
# print(results_path)

In [ ]:
results